# HanuAI – ML Assessment

## Task 1: Customer Review Web Scraping and Sentiment Analysis

### Objective
Inspect product reviews, prepare structured review data, analyze customer sentiment, identify recurring feedback themes, document collection challenges, and provide business recommendations.

### Product
JLab JBuds Lux Over-Ear Noise Cancelling True Wireless Headphones – Mauve

[View the Best Buy Canada product page](https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy/18145791)

### Workflow
1. Import libraries and load the available review dataset.
2. Validate data quality and required columns.
3. Inspect the product page and available review data.
4. Perform sentiment analysis using VADER.
5. Analyze recurring customer feedback themes.
6. Summarize results and prepare business recommendations.
7. Document collection limitations and verify deliverables.

### Data Provenance Note
The existing 50-row analysis dataset and the 10 records extracted from the inspected live-page state are separate datasets. The origin of the 50-row dataset has not been independently verified as a 50-review scrape from Best Buy Canada.

In [1]:
# HanuAI – ML Assessment Task 1
# Step 1: Import libraries and load the customer review dataset

import pandas as pd
from pathlib import Path

# Define the project paths
PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "data" / "hanuaI_reviews.xlsx"

# Load the dataset
df = pd.read_excel(DATA_FILE)

# Display basic dataset information
print("Dataset loaded successfully!")
print("Total reviews:", len(df))
print("Column names:", df.columns.tolist())

# Display the first five reviews
df.head()

Dataset loaded successfully!
Total reviews: 50
Column names: ['ID', 'Review title', 'Reviewer', 'Date', 'Rating', 'Review Text', 'Source']


,ID,Review title,Reviewer,Date,Rating,Review Text,Source
0,REV001,Lux ANC Review - Great Color and Comfortable,Schulzy,2024-04-04,5,"Comfortable fit, attractive color, useful butt...",jlabaudio.com
1,REV002,THE MVP HEADPHONE!,TheRogueOne,2024-04-06,4,Positive overall experience.,jlabaudio.com
2,REV003,Great headphones on a budget,Jfrias1,2024-05-28,5,Good headphones after adjusting the EQ settings.,jlabaudio.com
3,REV004,Perfect for Workouts,DanitaB,2026-01-21,5,"Good battery, comfortable for workouts, fits w...",bestbuy.com
4,REV005,"Great Headphones, Great sound.",Isaiah S,2026-10-23,4,"Good sound, noise cancellation, listening mode...",jlabaudio.com


In [2]:
# Step 2: Validate the customer review dataset

print("Total reviews:", len(df))

# Check for missing values
print("\nMissing values in each column:")
print(df.isnull().sum())

# Check for duplicate review IDs
print("\nDuplicate IDs:", df["ID"].duplicated().sum())

# Check for invalid ratings
print(
    "Invalid ratings:",
    ((df["Rating"] < 1) | (df["Rating"] > 5) | df["Rating"].isnull()).sum()
)

# Check for empty review texts
print(
    "Empty review texts:",
    (df["Review Text"].isna() | df["Review Text"].astype(str).str.strip().eq("")).sum()
)

# Check for duplicate review texts
print(
    "Duplicate review texts:",
    df["Review Text"].duplicated().sum()
)

Total reviews: 50

Missing values in each column:
ID              0
Review title    0
Reviewer        0
Date            0
Rating          0
Review Text     0
Source          0
dtype: int64

Duplicate IDs: 0
Invalid ratings: 0
Empty review texts: 0
Duplicate review texts: 0


In [3]:
# Step 3: Perform sentiment analysis using VADER

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Calculate sentiment scores for each review
df["Sentiment Score"] = df["Review Text"].apply(
    lambda text: analyzer.polarity_scores(str(text))["compound"]
)

# Classify each review based on its compound score
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["Sentiment"] = df["Sentiment Score"].apply(classify_sentiment)

# Display the sentiment distribution
print("Sentiment Distribution:")
print(df["Sentiment"].value_counts())

# Display a sample of the results
df[["ID", "Review title", "Rating", "Sentiment Score", "Sentiment"]].head(10)

Sentiment Distribution:
Sentiment
Positive    37
Negative    10
Neutral      3
Name: count, dtype: int64


,ID,Review title,Rating,Sentiment Score,Sentiment
0,REV001,Lux ANC Review - Great Color and Comfortable,5,0.9042,Positive
1,REV002,THE MVP HEADPHONE!,4,0.5574,Positive
2,REV003,Great headphones on a budget,5,0.4404,Positive
3,REV004,Perfect for Workouts,5,0.7964,Positive
4,REV005,"Great Headphones, Great sound.",4,0.4404,Positive
5,REV006,Only a week old,5,0.6597,Positive
6,REV007,Good quality,5,0.4215,Positive
7,REV008,Worth it,5,0.3919,Positive
8,REV009,Affordable headphones,4,0.7430,Positive
9,REV010,JLab Lux ANC,4,0.9081,Positive


In [4]:
# Step 4: Save sentiment analysis results

from pathlib import Path

# Create the reports folder if it does not exist
REPORTS_DIR = Path.cwd() / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Save results to Excel
output_file = REPORTS_DIR / "sentiment_results.xlsx"
df.to_excel(output_file, index=False)

print("Sentiment results saved successfully!")
print("File location:", output_file)

Sentiment results saved successfully!
File location: C:\Users\sridh\HanuAI_Task1\reports\sentiment_results.xlsx


In [5]:
# Step 5: Analyze customer feedback themes

import re
from collections import Counter

# Define the customer feedback themes and their keywords
themes = {
    "Sound Quality": ["sound", "audio", "bass", "volume", "quality"],
    "Comfort and Fit": ["comfortable", "comfort", "fit", "ears", "head"],
    "Noise Cancellation": ["noise cancellation", "anc", "noise cancelling"],
    "Battery Life": ["battery", "hours", "life", "lasting"],
    "Charging": ["charge", "charging", "charger"],
    "Microphone and Calls": ["microphone", "mic", "call", "voice"],
    "Build Quality and Durability": ["build", "quality", "crack", "broken", "durability"],
    "Price and Value": ["price", "budget", "affordable", "value", "worth"],
    "Appearance and Color": ["color", "colour", "design", "look", "appearance"],
    "Connectivity": ["bluetooth", "connect", "connection", "pairing"]
}

# Find reviews mentioning each theme
theme_results = []

for theme, keywords in themes.items():
    matching_reviews = df[
        df["Review Text"].str.lower().apply(
            lambda text: any(keyword in text for keyword in keywords)
        )
    ]

    theme_results.append({
        "Theme": theme,
        "Reviews Mentioning Theme": len(matching_reviews),
        "Positive": (matching_reviews["Sentiment"] == "Positive").sum(),
        "Negative": (matching_reviews["Sentiment"] == "Negative").sum(),
        "Neutral": (matching_reviews["Sentiment"] == "Neutral").sum()
    })

theme_df = pd.DataFrame(theme_results)

# Display the results
theme_df

,Theme,Reviews Mentioning Theme,Positive,Negative,Neutral
0,Sound Quality,18,15,2,1
1,Comfort and Fit,17,13,4,0
2,Noise Cancellation,11,7,4,0
3,Battery Life,4,3,1,0
4,Charging,2,1,1,0
5,Microphone and Calls,6,2,2,2
6,Build Quality and Durability,11,6,4,1
7,Price and Value,10,8,1,1
8,Appearance and Color,5,5,0,0
9,Connectivity,4,2,1,1


In [6]:
# Step 6: Verify and display customer theme counts

print(theme_df.to_string(index=False))

# Check whether sentiment counts match the number of reviews per theme
theme_df["Sentiment Count Total"] = (
    theme_df["Positive"]
    + theme_df["Negative"]
    + theme_df["Neutral"]
)

theme_df["Counts Match"] = (
    theme_df["Reviews Mentioning Theme"]
    == theme_df["Sentiment Count Total"]
)

display(theme_df)

                       Theme  Reviews Mentioning Theme  Positive  Negative  Neutral
               Sound Quality                        18        15         2        1
             Comfort and Fit                        17        13         4        0
          Noise Cancellation                        11         7         4        0
                Battery Life                         4         3         1        0
                    Charging                         2         1         1        0
        Microphone and Calls                         6         2         2        2
Build Quality and Durability                        11         6         4        1
             Price and Value                        10         8         1        1
        Appearance and Color                         5         5         0        0
                Connectivity                         4         2         1        1


,Theme,Reviews Mentioning Theme,Positive,Negative,Neutral,Sentiment Count Total,Counts Match
0,Sound Quality,18,15,2,1,18,True
1,Comfort and Fit,17,13,4,0,17,True
2,Noise Cancellation,11,7,4,0,11,True
3,Battery Life,4,3,1,0,4,True
4,Charging,2,1,1,0,2,True
5,Microphone and Calls,6,2,2,2,6,True
6,Build Quality and Durability,11,6,4,1,11,True
7,Price and Value,10,8,1,1,10,True
8,Appearance and Color,5,5,0,0,5,True
9,Connectivity,4,2,1,1,4,True


In [7]:
# Step 7: Save customer theme analysis

theme_df.to_excel(
    REPORTS_DIR / "customer_theme_analysis.xlsx",
    index=False
)

print("Customer theme analysis saved successfully!")
print("File:", REPORTS_DIR / "customer_theme_analysis.xlsx")

Customer theme analysis saved successfully!
File: C:\Users\sridh\HanuAI_Task1\reports\customer_theme_analysis.xlsx


In [8]:
# Step 8: Generate the final project summary

print("=" * 45)
print("HANUAI – ML ASSESSMENT TASK 1")
print("CUSTOMER REVIEW ANALYSIS SUMMARY")
print("=" * 45)

print("\n1. DATASET")
print("Total reviews analyzed:", len(df))
print("Missing values:", int(df.isnull().sum().sum()))

print("\n2. SENTIMENT DISTRIBUTION")
sentiment_counts = df["Sentiment"].value_counts()

for sentiment in ["Positive", "Negative", "Neutral"]:
    count = int(sentiment_counts.get(sentiment, 0))
    percentage = count / len(df) * 100
    print(f"{sentiment}: {count} reviews ({percentage:.1f}%)")

print("\n3. CUSTOMER FEEDBACK THEMES")
display(theme_df)

print("\n4. OUTPUT FILES")
print("Sentiment results: reports/sentiment_results.xlsx")
print("Theme analysis: reports/customer_theme_analysis.xlsx")

print("\nAnalysis completed.")

HANUAI – ML ASSESSMENT TASK 1
CUSTOMER REVIEW ANALYSIS SUMMARY

1. DATASET
Total reviews analyzed: 50
Missing values: 0

2. SENTIMENT DISTRIBUTION
Positive: 37 reviews (74.0%)
Negative: 10 reviews (20.0%)
Neutral: 3 reviews (6.0%)

3. CUSTOMER FEEDBACK THEMES


,Theme,Reviews Mentioning Theme,Positive,Negative,Neutral,Sentiment Count Total,Counts Match
0,Sound Quality,18,15,2,1,18,True
1,Comfort and Fit,17,13,4,0,17,True
2,Noise Cancellation,11,7,4,0,11,True
3,Battery Life,4,3,1,0,4,True
4,Charging,2,1,1,0,2,True
5,Microphone and Calls,6,2,2,2,6,True
6,Build Quality and Durability,11,6,4,1,11,True
7,Price and Value,10,8,1,1,10,True
8,Appearance and Color,5,5,0,0,5,True
9,Connectivity,4,2,1,1,4,True



4. OUTPUT FILES
Sentiment results: reports/sentiment_results.xlsx
Theme analysis: reports/customer_theme_analysis.xlsx

Analysis completed.


In [9]:
# Final source verification
print("Review counts by source:")
print(df["Source"].value_counts())

print("\nUnique source values:")
print(df["Source"].unique())

print("\nSample review IDs and sources:")
display(df[["ID", "Review title", "Source"]].head(10))

Review counts by source:
Source
jlabaudio.com    49
bestbuy.com       1
Name: count, dtype: int64

Unique source values:
<StringArray>
['jlabaudio.com', 'bestbuy.com']
Length: 2, dtype: str

Sample review IDs and sources:


,ID,Review title,Source
0,REV001,Lux ANC Review - Great Color and Comfortable,jlabaudio.com
1,REV002,THE MVP HEADPHONE!,jlabaudio.com
2,REV003,Great headphones on a budget,jlabaudio.com
3,REV004,Perfect for Workouts,bestbuy.com
4,REV005,"Great Headphones, Great sound.",jlabaudio.com
5,REV006,Only a week old,jlabaudio.com
6,REV007,Good quality,jlabaudio.com
7,REV008,Worth it,jlabaudio.com
8,REV009,Affordable headphones,jlabaudio.com
9,REV010,JLab Lux ANC,jlabaudio.com


In [10]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [11]:
# HanuAI Task 1: Inspect the public product page
# This checks the page response; it does not bypass CAPTCHA or access controls.

import requests
from bs4 import BeautifulSoup

product_url = (
    "https://www.bestbuy.ca/en-ca/product/"
    "jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve/"
    "18145791"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/130.0.0.0 Safari/537.36"
}

try:
    response = requests.get(
        product_url,
        headers=headers,
        timeout=20
    )

    print("HTTP status:", response.status_code)
    print("Final URL:", response.url)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text(" ", strip=True)

        print("\nPage title:", soup.title.get_text(strip=True) if soup.title else "Not found")
        print("HTML length:", len(response.text))
        print("Page mentions reviews:", "review" in page_text.lower())

        # Show a short excerpt containing the word "review", if present.
        position = page_text.lower().find("review")
        if position >= 0:
            print("\nReview-related excerpt:")
            print(page_text[max(0, position - 200):position + 500])
        else:
            print("\nNo review text found in the returned HTML.")
    else:
        print("The page did not return a normal success response.")

except requests.RequestException as error:
    print("Request failed:", error)

HTTP status: 403
Final URL: https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve/18145791
The page did not return a normal success response.


In [12]:
# HanuAI Task 1: Document review-page inspection
# The product page returned HTTP 403 to a direct Python request.
# The following review controls were observed manually in Chrome.
# This documents the inspection; it does not claim reviews were scraped.

review_sort_options = [
    "Most Relevant",
    "Most Helpful",
    "Newest",
    "Oldest",
    "High Rating",
    "Low Rating"
]

scraping_status = {
    "Website": "Best Buy Canada",
    "Product": "JLab JBuds Lux Over-Ear Noise Cancelling Headphones",
    "Direct HTTP request": "403 Forbidden",
    "Review section visible in browser": True,
    "Show More control": "Visible",
    "Available sorting options": ", ".join(review_sort_options),
    "Automated review extraction": "Not completed"
}

for key, value in scraping_status.items():
    print(f"{key}: {value}")

Website: Best Buy Canada
Product: JLab JBuds Lux Over-Ear Noise Cancelling Headphones
Direct HTTP request: 403 Forbidden
Review section visible in browser: True
Show More control: Visible
Available sorting options: Most Relevant, Most Helpful, Newest, Oldest, High Rating, Low Rating
Automated review extraction: Not completed


In [13]:
# HanuAI Task 1: Save the scraping inspection log

inspection_df = pd.DataFrame([
    {"Item": key, "Result": value}
    for key, value in scraping_status.items()
])

inspection_file = REPORTS_DIR / "scraping_inspection.xlsx"
inspection_df.to_excel(inspection_file, index=False)

print("Inspection log saved:", inspection_file)
display(inspection_df)

Inspection log saved: C:\Users\sridh\HanuAI_Task1\reports\scraping_inspection.xlsx


,Item,Result
0,Website,Best Buy Canada
1,Product,JLab JBuds Lux Over-Ear Noise Cancelling Headp...
2,Direct HTTP request,403 Forbidden
3,Review section visible in browser,True
4,Show More control,Visible
5,Available sorting options,"Most Relevant, Most Helpful, Newest, Oldest, H..."
6,Automated review extraction,Not completed


In [14]:
# HanuAI Task 1: Generate the stakeholder report

from pathlib import Path

report = """
HANUAI – ML ASSESSMENT
TASK 1: WEB SCRAPING AND SENTIMENT ANALYSIS

1. OBJECTIVE
Analyze customer reviews for the JLab JBuds Lux Over-Ear Noise
Cancelling Headphones and identify customer satisfaction drivers,
complaints, and potential product improvements.

2. DATASET AND VALIDATION
The working dataset contains 50 reviews and seven columns:
ID, Review title, Reviewer, Date, Rating, Review Text, and Source.
Validation found no missing values, duplicate IDs, invalid ratings,
empty review texts, or duplicate review texts.

Source limitation:
The dataset lists 49 reviews as jlabaudio.com and one as
bestbuy.com. Therefore, it cannot be represented as 50 reviews
collected from Best Buy Canada.

3. SCRAPING ATTEMPT AND CHALLENGES
The Best Buy Canada product page was inspected in a browser.
The review section, Show More control, and these sorting options
were visible: Most Relevant, Most Helpful, Newest, Oldest,
High Rating, and Low Rating.

A direct Python HTTP request returned status 403 Forbidden.
Automated review extraction was not completed. No attempt was made
to bypass the site's access restrictions. A permitted collection
method or authorization is needed to complete this part.

4. SENTIMENT ANALYSIS
VADER was applied to the review text using compound sentiment scores:
- Positive: 37 reviews (74%)
- Negative: 10 reviews (20%)
- Neutral: 3 reviews (6%)

VADER is a rule-based method, not a model trained on this dataset.
Its classifications should be reviewed against the original text.

5. CUSTOMER FEEDBACK THEMES
The keyword-based theme analysis identified the following themes:
- Sound Quality: 18 reviews
- Comfort and Fit: 17 reviews
- Noise Cancellation: 11 reviews
- Build Quality and Durability: 11 reviews
- Price and Value: 10 reviews
- Microphone and Calls: 6 reviews
- Appearance and Color: 5 reviews
- Battery Life: 4 reviews
- Connectivity: 4 reviews
- Charging: 2 reviews

A review may mention multiple themes, so theme counts overlap.
Keyword matching identifies mentions, not necessarily positive or
negative opinions about a feature.

6. POTENTIAL BUSINESS INSIGHTS AND RECOMMENDATIONS
- Sound quality and comfort appear frequently in the feedback.
  Continue monitoring these areas and preserve features customers value.
- Noise cancellation and build quality include negative classifications.
  Review the underlying complaints to identify recurring problems.
- Microphone and connectivity feedback warrants further investigation.
  Test call clarity, pairing stability, and performance across devices.
- Examine battery and charging complaints for recurring patterns.
- Validate these potential insights by manually reviewing representative
  positive and negative reviews before making product decisions.

7. LIMITATIONS AND NEXT STEPS
The sample's Best Buy Canada origin is not verified, and automated
scraping remains incomplete. Obtain an approved collection method,
collect at least 50 genuine reviews from the required source, and rerun
the analysis before presenting results as Best Buy-specific findings.

8. TOOLS
Python, pandas, openpyxl, VADER Sentiment Analysis, and
keyword-based text analysis.

CONCLUSION
The current analysis demonstrates dataset validation, sentiment
classification, and preliminary customer-theme analysis. The scraping
requirement remains incomplete, so the results should be treated as
preliminary rather than a completed Best Buy Canada scraping study.
"""

report_path = REPORTS_DIR / "HanuAI_Task1_Report.txt"
report_path.write_text(report.strip(), encoding="utf-8")

print("Report saved successfully:", report_path)

Report saved successfully: C:\Users\sridh\HanuAI_Task1\reports\HanuAI_Task1_Report.txt


In [15]:
# HanuAI Task 1: Final deliverable check

required_files = [
    PROJECT_DIR / "HanuAI_Task1.ipynb",
    PROJECT_DIR / "data" / "hanuaI_reviews.xlsx",
    REPORTS_DIR / "sentiment_results.xlsx",
    REPORTS_DIR / "customer_theme_analysis.xlsx",
    REPORTS_DIR / "scraping_inspection.xlsx",
    REPORTS_DIR / "HanuAI_Task1_Report.txt",
]

print("FINAL FILE CHECK")
print("-" * 40)

for file_path in required_files:
    status = "FOUND" if file_path.exists() else "MISSING"
    print(f"{status}: {file_path.name}")

FINAL FILE CHECK
----------------------------------------
FOUND: HanuAI_Task1.ipynb
FOUND: hanuaI_reviews.xlsx
FOUND: sentiment_results.xlsx
FOUND: customer_theme_analysis.xlsx
FOUND: scraping_inspection.xlsx
FOUND: HanuAI_Task1_Report.txt


In [16]:
%pip install selenium

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/11.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.8 MB 4.2 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/11.8 MB 3.9 MB/s eta 0:00:03
   -------- ------------------------------- 2.4/11.8 MB 4.3 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/11.8 MB 4.1 MB/s eta 0:00:03
   -------------- ------------------------- 4.2/11.8 MB 4.3 MB/s eta 0:00:02
   ----------------- ---------------------- 5.2/11.8 MB 4.6 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.8 MB 4.8 MB/s eta 0:00:02
   -------------------------- ------------- 7.9/11.8 MB 4.9 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.8 MB 5.1 MB/s eta 0:00:01
   ----------------------------------- ---- 10.5/11.8 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------  11.5/11.8 MB 5.4 MB/s eta 0:00:01
   --------------

In [17]:
# HanuAI Task 1: Open the Best Buy product page using Selenium

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

product_url = (
    "https://www.bestbuy.ca/en-ca/product/"
    "jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve/"
    "18145791"
)

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
driver.get(product_url)

print("Browser opened:", driver.title)
print("Current URL:", driver.current_url)

Browser opened: JLab JBuds LUX Over-Ear Noise Cancelling True Wireless Headphones - Mauve - Only at Best Buy | Best Buy Canada
Current URL: https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy/18145791


In [18]:

# Check whether review content is present on the loaded page
from selenium.webdriver.common.by import By

print("Page title:", driver.title)

page_text = driver.find_element(By.TAG_NAME, "body").text

for keyword in ["Reviews", "Show More", "Most Helpful", "Newest", "High Rating", "Low Rating"]:
    print(f"{keyword}: {'Found' if keyword.lower() in page_text.lower() else 'Not found'}")

print("\n--- Review-related text preview ---")
for line in page_text.splitlines():
    if any(word in line.lower() for word in ["review", "show more", "rating"]):
        print(line[:250])


Page title: JLab JBuds LUX Over-Ear Noise Cancelling True Wireless Headphones - Mauve - Only at Best Buy | Best Buy Canada
Reviews: Found
Show More: Not found
Most Helpful: Not found
Newest: Not found
High Rating: Not found
Low Rating: Not found

--- Review-related text preview ---
This description was generated by AI and reviewed for accuracy.
4.6(4015 Reviews)
Customer Reviews
(4015 Reviews)
This description was generated by AI and reviewed for accuracy.
Reviews


In [19]:

# Inspect review-related elements loaded by the browser
from selenium.webdriver.common.by import By

for selector in [
    '[data-testid*="review"]',
    '[class*="review"]',
    '[id*="review"]',
    'button'
]:
    elements = driver.find_elements(By.CSS_SELECTOR, selector)
    print(f"\nSelector: {selector} | Elements found: {len(elements)}")

    for element in elements[:8]:
        text = element.text.strip().replace("\n", " ")
        if text:
            print(" ", text[:180])



Selector: [data-testid*="review"] | Elements found: 1

Selector: [class*="review"] | Elements found: 9
  4.6(4015 Reviews)
  4.6(4015 Reviews)
  (4015 Reviews)
  (4015 Reviews)
  (4015 Reviews)

Selector: [id*="review"] | Elements found: 2
  Customer Reviews (4015 Reviews)

Selector: button | Elements found: 94
  Shop


In [20]:

# Scroll to the Customer Reviews section normally
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

review_section = driver.find_element(By.ID, "reviews")
driver.execute_script(
    "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});",
    review_section
)

print("Review section found:", review_section.text[:500])

# Check the section again after scrolling
import time
time.sleep(3)

page_text = driver.find_element(By.TAG_NAME, "body").text
print("\n--- Review controls after scrolling ---")

for keyword in ["Show More", "Most Helpful", "Most Relevant", "Newest", "High Rating", "Low Rating"]:
    print(f"{keyword}: {'Found' if keyword.lower() in page_text.lower() else 'Not found'}")


Review section found: Customer Reviews
(4015 Reviews)

--- Review controls after scrolling ---
Show More: Not found
Most Helpful: Not found
Most Relevant: Not found
Newest: Not found
High Rating: Not found
Low Rating: Not found


In [21]:

# Search the loaded HTML for review-related content
html = driver.page_source

for keyword in [
    "reviewText",
    "reviewTitle",
    "reviewerName",
    "reviewBody",
    "Show More",
    "Most Helpful"
]:
    print(f"{keyword}: {'Found' if keyword.lower() in html.lower() else 'Not found'}")

print("\nHTML length:", len(html))


reviewText: Not found
reviewTitle: Not found
reviewerName: Found
reviewBody: Found
Show More: Not found
Most Helpful: Not found

HTML length: 1069707


In [22]:

# Inspect small snippets around review-related field names
for keyword in ["reviewerName", "reviewBody"]:
    print(f"\n--- {keyword} ---")
    start = 0
    count = 0

    while count < 3:
        position = html.lower().find(keyword.lower(), start)
        if position == -1:
            break

        print(html[max(0, position - 200):position + 500])
        start = position + len(keyword)
        count += 1



--- reviewerName ---
2e8d","thumbnailUrl":"https://photos-us.bazaarvoice.com/photo/2/cGhvdG86amxhYmF1ZGlvLWNvbnY/96014819-f730-53db-927e-792a2c04291c"}],"rating":4,"reviewSource":"ALL","reviewerLocation":"jlabaudio.com","reviewerName":"Schulzy","submissionTime":"2024-04-04T01:59:02","syndicationSource":{"logoImageUrl":"https://photos-us.bazaarvoice.com/photo/2/cGhvdG86YXR0cmlidXRpb25sb2dvMg/c1a2c459-3893-4e71-b0fc-f2a4b8c4324f","name":"jlabaudio.com"},"title":"Lux ANC Review - Great Color and Comfortable","totalNegativeFeedbackCount":0,"totalPositiveFeedbackCount":0},{"comment":"JLAB never disappoints. This is my third pair of over ear headphones, and these are the most solid of the ones I’ve had. The sound qual
09cf","thumbnailUrl":"https://photos-us.bazaarvoice.com/photo/2/cGhvdG86amxhYmF1ZGlvLWNvbnY/4c5639ea-7e71-5f90-bc8e-9883c0bf73c3"}],"rating":5,"reviewSource":"ALL","reviewerLocation":"jlabaudio.com","reviewerName":"TheRogueOne","submissionTime":"2024-04-06T16:14:42","syndicati

In [23]:

from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

for i, script in enumerate(soup.find_all("script")):
    content = script.string or script.get_text()
    
    if "reviewerName" in content or '"reviewBody"' in content:
        print("Script number:", i)
        print("Script ID:", script.get("id"))
        print("Script type:", script.get("type"))
        print("Content length:", len(content))
        print("Starts with:", content[:250])
        print("-" * 50)


Script number: 21
Script ID: product-json-ld
Script type: application/ld+json
Content length: 3212
Starts with: {"@context":"http://schema.org/","@type":"Product","brand":{"@type":"Brand","name":"JLAB"},"gtin12":["810119071545"],"name":"JLab JBuds LUX Over-Ear Noise Cancelling True Wireless Headphones - Mauve - Only at Best Buy","model":"JBuds Lux ANC","sku":"
--------------------------------------------------
Script number: 39
Script ID: None
Script type: None
Content length: 114678
Starts with: 
        window.__INITIAL_STATE__ = {"addOnsPage":{"availableWarranties":[],"availableWarrantiesLoading":true,"warrantyBenefitsMessage":{"18145791":{}},"parentItemJustAdded":false},"ads":{"googleAds":{},"criteoAds":{}},"analytics":{"marketingParams":
--------------------------------------------------


In [24]:

# Inspect the initial-state data for review records
import re

state_script = soup.find(
    "script",
    string=lambda text: text and "window.__INITIAL_STATE__" in text
)

if state_script:
    state_text = state_script.string or state_script.get_text()
    print("Initial state found:", True)
    print("Contains reviewerName:", "reviewerName" in state_text)
    print("Contains reviewBody:", "reviewBody" in state_text)
    print("Contains reviewSource:", "reviewSource" in state_text)
    print("Contains submissionTime:", "submissionTime" in state_text)
    print("Contains review records:", '"comment"' in state_text)
else:
    print("Initial state script not found")


Initial state found: True
Contains reviewerName: True
Contains reviewBody: False
Contains reviewSource: True
Contains submissionTime: True
Contains review records: True


In [25]:

import json

# Extract the JSON object assigned to window.__INITIAL_STATE__
match = re.search(
    r"window\.__INITIAL_STATE__\s*=\s*(\{.*\})\s*;?\s*$",
    state_text,
    re.DOTALL
)

if match:
    try:
        initial_state = json.loads(match.group(1))
        print("JSON parsed successfully")
        print("Top-level keys:", list(initial_state.keys()))
    except json.JSONDecodeError as e:
        print("JSON parsing failed:", e)
else:
    print("Could not isolate the initial-state JSON")


JSON parsing failed: Extra data: line 1 column 88619 (char 88618)


In [26]:

import json

# Find the opening brace after the initial-state assignment
start = state_text.find("=", state_text.find("window.__INITIAL_STATE__")) + 1

while start < len(state_text) and state_text[start].isspace():
    start += 1

# Find the end of the JSON object while respecting strings and escapes
depth = 0
in_string = False
escaped = False
end = None

for i in range(start, len(state_text)):
    char = state_text[i]

    if in_string:
        if escaped:
            escaped = False
        elif char == "\\":
            escaped = True
        elif char == '"':
            in_string = False
    else:
        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

if end is not None:
    initial_state = json.loads(state_text[start:end])
    print("JSON parsed successfully")
    print("Top-level keys:", list(initial_state.keys()))
else:
    print("Could not find the end of the JSON object")


JSON parsed successfully
Top-level keys: ['addOnsPage', 'ads', 'analytics', 'app', 'bazaarVoiceJS', 'bbProtectionMembershipDialog', 'bbp', 'brand', 'cart', 'cartCategoryIds', 'cartPage', 'categoryContent', 'config', 'content', 'dynamicContent', 'errors', 'featuredSkuList', 'help', 'helpSearch', 'homePage', 'intl', 'manufacturerWarranty', 'navigation', 'notification', 'npsSurvey', 'offers', 'product', 'productList', 'productRelatedProducts', 'productSellers', 'promotionalBadges', 'recommendations', 'relatedContentLinks', 'requiredProducts', 'routing', 'search', 'sellers', 'servicePlan', 'servicePlanBenefits', 'sideNavigation', 'skuList', 'storeMessages', 'stores', 'storesStatus', 'user']


In [27]:

# Inspect likely locations for review data
for section_name in ["bazaarVoiceJS", "product"]:
    section = initial_state.get(section_name, {})

    print(f"\n--- {section_name} ---")
    print("Type:", type(section).__name__)

    if isinstance(section, dict):
        print("Keys:", list(section.keys())[:60])
    else:
        print(str(section)[:1000])



--- bazaarVoiceJS ---
Type: dict
Keys: ['isloadedProductReviewsJS']

--- product ---
Type: dict
Keys: ['hasMoreOffersOnLogin', 'isAvailabilityError', 'availability', 'isAvailabilityLoading', 'customerReviews', 'category', 'loadingProduct', 'isCategoryLoading', 'loadingStores', 'loadingVariants', 'onLoginCustGroupIds', 'productVariantError', 'protectionPlanSheetVariant', 'product', 'recommendations', 'productConditions', 'specialOffers', 'targettedContent', 'pdpFindingMethod', 'whichMediaGalleryModalOpen', 'warrantyAvailabilities', 'warrantyAvailabilityStatus', 'badges']


In [28]:

# Inspect the customerReviews structure
reviews_data = initial_state["product"]["customerReviews"]

print("Type:", type(reviews_data).__name__)

if isinstance(reviews_data, dict):
    print("Keys:", list(reviews_data.keys()))
    for key, value in reviews_data.items():
        print(f"\n{key}: type={type(value).__name__}")
        if isinstance(value, (dict, list)):
            print("Preview:", str(value)[:500])
        else:
            print("Value:", str(value)[:300])
elif isinstance(reviews_data, list):
    print("Number of items:", len(reviews_data))
    print("Preview:", str(reviews_data[:2])[:1000])
else:
    print("Value:", str(reviews_data)[:1000])


Type: dict
Keys: ['displayReviews', 'customerReviewsWithPhotosOnly', 'ratingSummary']

displayReviews: type=dict
Preview: {'customerReviews': [{'comment': 'I received the Lux ANC headphones through the Amazon Vine program.  They paired quickly to my Motorola G 5G (2023) phone with Google Fast pair and also with my Chromebook.   \n\nI love the Cloud white/grey color.  The ear cups have a unique egg shape.  They extend and fit my 23" head with room to spare.  The headband and pads are very comfortable.  \n\nThe control buttons are very large and easy to operate by touch.  I like the voice prompts for power, connect, 

customerReviewsWithPhotosOnly: type=dict
Preview: {'status': 'idle', 'sortOption': 'relevancy', 'currentPage': 0, 'pageSize': 10, 'totalPages': 0, 'totalRows': 0, 'loadingReviews': False, 'customerReviews': [], 'filter': {'hasPhotos': True}}

ratingSummary: type=dict
Preview: {'averageRating': 4.63, 'reviewCount': 4015, 'ratingDistribution': {'1': 53, '2': 48, '3': 156, '4'

In [29]:

import pandas as pd

# Get the review records embedded in the page state
reviews = (
    initial_state["product"]["customerReviews"]
    ["displayReviews"]["customerReviews"]
)

print("Records available:", len(reviews))
print("Fields in first record:", list(reviews[0].keys()) if reviews else "No records found")

# Convert the available records into the required assessment columns
rows = []

for review in reviews:
    rows.append({
        "ID": review.get("id"),
        "Review title": review.get("title"),
        "Review Text": review.get("comment"),
        "Date": (
            review.get("submissionTime", "")[:10]
            if review.get("submissionTime") else None
        ),
        "Rating": review.get("rating"),
        "Reviewer": review.get("reviewerName"),
        "Source": (
            review.get("syndicationSource", {}).get("name")
            or review.get("reviewerLocation")
            or review.get("reviewSource")
        )
    })

reviews_df = pd.DataFrame(rows)

print("\nExtracted shape:", reviews_df.shape)
display(reviews_df.head(5))
print("\nSource counts:")
print(reviews_df["Source"].value_counts(dropna=False))


Records available: 10
Fields in first record: ['comment', 'id', 'isVerifiedPurchaser', 'keyConsiderations', 'photos', 'rating', 'reviewSource', 'reviewerLocation', 'reviewerName', 'submissionTime', 'syndicationSource', 'title', 'totalNegativeFeedbackCount', 'totalPositiveFeedbackCount']

Extracted shape: (10, 7)


,ID,Review title,Review Text,Date,Rating,Reviewer,Source
0,1159371677,Lux ANC Review - Great Color and Comfortable,I received the Lux ANC headphones through the ...,2024-04-04,4,Schulzy,jlabaudio.com
1,1159550039,THE MVP HEADPHONE!,JLAB never disappoints. This is my third pair ...,2024-04-06,5,TheRogueOne,jlabaudio.com
2,1163555106,Great headphones on a budget,"I bought them a week ago, i had to tweak the E...",2024-05-28,4,Jfrias1,jlabaudio.com
3,413861161,Perfect for Workouts,[This review was collected as part of a promot...,2026-01-21,5,DanitaB,bestbuy.com
4,1159337628,I like them a lot.,"Got them little less than a month, use them mo...",2024-04-03,4,LuzVal,jlabaudio.com



Source counts:
Source
jlabaudio.com    8
bestbuy.com      2
Name: count, dtype: int64


In [30]:
customer_reviews = initial_state["product"]["customerReviews"]

for section_name, section in customer_reviews.items():
    print(f"\nSection: {section_name}")

    if isinstance(section, dict):
        print("Available keys:", list(section.keys()))

        if "customerReviews" in section:
            print("Review records:", len(section["customerReviews"]))

        for key in ["currentPage", "pageSize", "totalPages", "totalRows", "loadingReviews"]:
            if key in section:
                print(f"{key}: {section[key]}")


Section: displayReviews
Available keys: ['customerReviews', 'currentPage', 'pageSize', 'totalPages', 'totalRows', 'sortOption', 'filter', 'loadingReviews']
Review records: 10
currentPage: 1
pageSize: 10
totalPages: 402
totalRows: 4014
loadingReviews: False

Section: customerReviewsWithPhotosOnly
Available keys: ['status', 'sortOption', 'currentPage', 'pageSize', 'totalPages', 'totalRows', 'loadingReviews', 'customerReviews', 'filter']
Review records: 0
currentPage: 0
pageSize: 10
totalPages: 0
totalRows: 0
loadingReviews: False

Section: ratingSummary
Available keys: ['averageRating', 'reviewCount', 'ratingDistribution', 'localeReviewCount', 'recommendationCount', 'keyConsiderations', 'sentimentSummary', 'photoCountFirstPage', 'hasPhotos']


In [31]:
from selenium.webdriver.common.by import By

print("Current page:", driver.current_url)

# Check the main page and any embedded frames
driver.switch_to.default_content()

frames = driver.find_elements(By.TAG_NAME, "iframe")
print("Iframes found:", len(frames))

for i, frame in enumerate(frames):
    try:
        print(f"\nIframe {i}:")
        print("Title:", frame.get_attribute("title"))
        print("Source:", frame.get_attribute("src"))
    except Exception as e:
        print("Could not inspect iframe:", e)

# Check normal page buttons and links related to reviews
driver.switch_to.default_content()

for element in driver.find_elements(By.XPATH, "//*[self::button or self::a]"):
    try:
        text = element.text.strip()
        if any(word in text.lower() for word in
               ["show more", "load more", "next", "review", "helpful", "relevant", "newest"]):
            print("Control:", repr(text),
                  "| Tag:", element.tag_name,
                  "| Visible:", element.is_displayed(),
                  "| Enabled:", element.is_enabled())
    except Exception:
        pass

Current page: https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy/18145791
Iframes found: 7

Iframe 0:
Title: 3rd party ad content
Source: https://f573577e2959dbcd48cf1707e83d87f5.safeframe.googlesyndication.com/safeframe/1-0-45/html/container.html

Iframe 1:
Title: 
Source: https://chat.bestbuy.ca/chat/genesys-chat.html?lang=en

Iframe 2:
Title: Advertisement
Source: https://googleads.g.doubleclick.net/pagead/ads?client=ca-pub-4794720283461909&output=html&adk=1812271804&adf=3025194257&abgtt=6&lmt=1792937152&plaf=1%3A2%2C2%3A2%2C7%3A2&plat=1%3A128%2C2%3A128%2C3%3A128%2C4%3A128%2C8%3A128%2C9%3A32776%2C16%3A8388608%2C17%3A32%2C24%3A32%2C25%3A32%2C30%3A1081344%2C32%3A32%2C41%3A32%2C42%3A32%2C43%3A32%2C44%3A32&format=0x0&url=https%3A%2F%2Fwww.bestbuy.ca%2Fen-ca%2Fproduct%2Fjlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy%2F18145791&pra=5&aiof=14&asro=0&aimartd=4&aieuf=1&aicrs=1

In [32]:
import re

page_source = driver.page_source

print("Occurrences of reviewerName:", page_source.count('"reviewerName"'))
print("Occurrences of submissionTime:", page_source.count('"submissionTime"'))
print("Occurrences of review IDs:", len(re.findall(r'"id"\s*:\s*\d+', page_source)))

Occurrences of reviewerName: 10
Occurrences of submissionTime: 10
Occurrences of review IDs: 0


In [33]:
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

reviews_df.to_csv(
    output_dir / "bestbuy_page_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

reviews_df.to_excel(
    output_dir / "bestbuy_page_reviews.xlsx",
    index=False
)

print("Saved:", output_dir / "bestbuy_page_reviews.csv")
print("Saved:", output_dir / "bestbuy_page_reviews.xlsx")
print("Records saved:", len(reviews_df))

Saved: data\bestbuy_page_reviews.csv
Saved: data\bestbuy_page_reviews.xlsx
Records saved: 10


In [34]:
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
from pathlib import Path

# Download VADER's sentiment lexicon if it is not already available
nltk.download("vader_lexicon", quiet=True)

# Load the extracted reviews
df = pd.read_csv("data/bestbuy_page_reviews.csv")

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Calculate sentiment scores
df["Sentiment Score"] = df["Review Text"].fillna("").apply(
    lambda text: sia.polarity_scores(text)["compound"]
)

# Classify sentiment
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["Sentiment"] = df["Sentiment Score"].apply(classify_sentiment)

# Save results
Path("reports").mkdir(exist_ok=True)

df.to_csv("reports/bestbuy_reviews_sentiment.csv", index=False, encoding="utf-8-sig")
df.to_excel("reports/bestbuy_reviews_sentiment.xlsx", index=False)

# Display results
display(df[["ID", "Review title", "Rating", "Sentiment Score", "Sentiment", "Source"]])

print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())

print("\nSaved sentiment analysis files.")

,ID,Review title,Rating,Sentiment Score,Sentiment,Source
0,1159371677,Lux ANC Review - Great Color and Comfortable,4,0.9601,Positive,jlabaudio.com
1,1159550039,THE MVP HEADPHONE!,5,0.8500,Positive,jlabaudio.com
2,1163555106,Great headphones on a budget,4,0.6908,Positive,jlabaudio.com
3,413861161,Perfect for Workouts,5,0.8798,Positive,bestbuy.com
4,1159337628,I like them a lot.,4,0.7430,Positive,jlabaudio.com
5,1234783375,"Great Headphones, Great sound.",5,0.9893,Positive,jlabaudio.com
6,1216010829,I loved it so far but recently encountered an ...,3,0.7269,Positive,jlabaudio.com
7,1197128032,Great Sound!,5,0.9845,Positive,jlabaudio.com
8,1158090995,Read,5,0.8442,Positive,jlabaudio.com
9,379036967,JBud Lux ANC Wireless Headphones,5,0.9368,Positive,bestbuy.com



Sentiment distribution:
Sentiment
Positive    10
Name: count, dtype: int64

Saved sentiment analysis files.


In [35]:
import pandas as pd

df = pd.read_csv("reports/bestbuy_reviews_sentiment.csv")

print("Total reviews analyzed:", len(df))

print("\nRating distribution:")
print(df["Rating"].value_counts().sort_index())

print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())

print("\nAverage star rating:", round(df["Rating"].mean(), 2), "/ 5")

print("\nSentiment by star rating:")
print(pd.crosstab(df["Rating"], df["Sentiment"], margins=True))

print("\nSource distribution:")
print(df["Source"].value_counts())

Total reviews analyzed: 10

Rating distribution:
Rating
3    1
4    3
5    6
Name: count, dtype: int64

Sentiment distribution:
Sentiment
Positive    10
Name: count, dtype: int64

Average star rating: 4.5 / 5

Sentiment by star rating:
Sentiment  Positive  All
Rating                  
3                 1    1
4                 3    3
5                 6    6
All              10   10

Source distribution:
Source
jlabaudio.com    8
bestbuy.com      2
Name: count, dtype: int64


In [36]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("reports/bestbuy_reviews_sentiment.csv")

summary = pd.DataFrame({
    "Metric": [
        "Reviews analyzed",
        "Average star rating",
        "Positive sentiment",
        "Negative sentiment",
        "Neutral sentiment",
        "Reviews sourced from jlabaudio.com",
        "Reviews sourced from bestbuy.com"
    ],
    "Value": [
        len(df),
        round(df["Rating"].mean(), 2),
        int((df["Sentiment"] == "Positive").sum()),
        int((df["Sentiment"] == "Negative").sum()),
        int((df["Sentiment"] == "Neutral").sum()),
        int((df["Source"] == "jlabaudio.com").sum()),
        int((df["Source"] == "bestbuy.com").sum())
    ]
})

Path("reports").mkdir(exist_ok=True)
summary.to_excel("reports/review_analysis_summary.xlsx", index=False)

display(summary)
print("Saved: reports/review_analysis_summary.xlsx")

,Metric,Value
0,Reviews analyzed,10.0
1,Average star rating,4.5
2,Positive sentiment,10.0
3,Negative sentiment,0.0
4,Neutral sentiment,0.0
5,Reviews sourced from jlabaudio.com,8.0
6,Reviews sourced from bestbuy.com,2.0


Saved: reports/review_analysis_summary.xlsx


In [37]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("reports/bestbuy_reviews_sentiment.csv")

# Rating distribution
rating_counts = df["Rating"].value_counts().sort_index()

# Common words/themes for an initial inspection
keywords = {
    "Sound Quality": ["sound", "audio", "bass", "music"],
    "Comfort and Fit": ["comfort", "comfortable", "fit", "ear"],
    "Noise Cancellation": ["noise cancellation", "anc", "cancel"],
    "Battery Life": ["battery", "charge", "hours"],
    "Connectivity": ["bluetooth", "connect", "pair"],
    "Microphone and Calls": ["microphone", "mic", "call"]
}

theme_rows = []
for theme, words in keywords.items():
    matched = df["Review Text"].fillna("").str.lower().apply(
        lambda text: any(word in text for word in words)
    )
    theme_rows.append({
        "Theme": theme,
        "Reviews mentioning theme": int(matched.sum())
    })

theme_df = pd.DataFrame(theme_rows).sort_values(
    "Reviews mentioning theme", ascending=False
)

print("Rating distribution:")
print(rating_counts)

print("\nInitial keyword-based themes:")
display(theme_df)

print("\nPreliminary business insights:")
print("1. The average rating in this small sample is 4.5 out of 5.")
print("2. All 10 reviews received a positive VADER sentiment label.")
print("3. Most records are attributed to jlabaudio.com, so source attribution must be preserved.")
print("4. A larger sample is required before drawing reliable conclusions about customer preferences.")

theme_df.to_excel("reports/preliminary_theme_counts.xlsx", index=False)
print("\nSaved: reports/preliminary_theme_counts.xlsx")

Rating distribution:
Rating
3    1
4    3
5    6
Name: count, dtype: int64

Initial keyword-based themes:


,Theme,Reviews mentioning theme
0,Sound Quality,8
1,Comfort and Fit,8
2,Noise Cancellation,4
3,Battery Life,2
4,Connectivity,2
5,Microphone and Calls,0



Preliminary business insights:
1. The average rating in this small sample is 4.5 out of 5.
2. All 10 reviews received a positive VADER sentiment label.
3. Most records are attributed to jlabaudio.com, so source attribution must be preserved.
4. A larger sample is required before drawing reliable conclusions about customer preferences.

Saved: reports/preliminary_theme_counts.xlsx


In [38]:
from pathlib import Path

report = """
HANUAI – ML ASSESSMENT
Task 1: Customer Review Scraping and Sentiment Analysis

1. Objective
The objective was to collect product reviews, structure the review data,
perform sentiment analysis, identify recurring themes, and develop business
recommendations.

2. Product and Data Collection
Product: JLab JBuds Lux Over-Ear Noise Cancelling True Wireless Headphones – Mauve
Product page:
https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy/18145791

The page's embedded state provided 10 review records. Eight records were
attributed to jlabaudio.com and two to bestbuy.com. Source labels were preserved.
The page reported 4,014 English review records, but only 10 were extracted.

3. Data Fields
The extracted data contains review ID, review title, review text, date,
rating, reviewer name, and source.

4. Data Collection Challenges
- A direct HTTP request returned status 403.
- The browser loaded the product page, but only 10 review records were embedded
  in the inspected page state.
- The page reported additional reviews and pagination, but further records
  were not retrieved in this run.
- The normal review controls were not exposed clearly through the inspected DOM.
- The assessment's 50-review minimum has not yet been met.

No claim is made that all reported reviews were scraped. Access controls should
not be bypassed; additional records should be obtained through a permitted
interface or an authorized data source.

5. Sentiment Analysis
VADER, a rule-based sentiment analyzer, was used to calculate compound
sentiment scores. Scores >= 0.05 were labelled Positive, scores <= -0.05
Negative, and the remaining scores Neutral.

Results for the 10 extracted records:
- Positive: 10
- Negative: 0
- Neutral: 0
- Average star rating: 4.5 out of 5
- Rating 3: 1 review
- Rating 4: 3 reviews
- Rating 5: 6 reviews

These results describe only the extracted sample. Sentiment labels may differ
from star ratings because VADER analyzes wording rather than rating values.

6. Preliminary Theme Analysis
Keyword-based matching found:
- Sound Quality: 8 reviews
- Comfort and Fit: 8 reviews
- Noise Cancellation: 4 reviews
- Battery Life: 2 reviews
- Connectivity: 2 reviews
- Microphone and Calls: 0 reviews

Themes may overlap. Keyword matching does not establish whether each mention
is positive or negative, and the counts should be treated as preliminary.

7. Preliminary Business Insights and Recommendations
- Review wording in the available sample frequently mentions sound quality
  and comfort. These areas may be useful priorities for a larger analysis.
- Evaluate noise cancellation, battery life, and connectivity using a larger,
  representative set of reviews.
- Preserve review source information so syndicated reviews can be distinguished
  from reviews attributed directly to Best Buy.
- Compare sentiment labels with star ratings and manually inspect a sample
  of predictions to assess classification quality.
- Collect at least 50 reviews through an authorized source before presenting
  broader customer sentiment conclusions.

8. Deliverable Status
Extracted records and preliminary sentiment and theme outputs have been saved
as CSV and Excel files. The 50-review collection requirement and final
validation remain incomplete. Results should not be presented as a completed
50-review scraping exercise.
"""

Path("reports").mkdir(exist_ok=True)
Path("reports/HanuAI_Task1_Report.txt").write_text(report, encoding="utf-8")

print("Updated: reports/HanuAI_Task1_Report.txt")

Updated: reports/HanuAI_Task1_Report.txt


In [39]:
from pathlib import Path

for folder in ["data", "reports"]:
    print(f"\n{folder}/")
    path = Path(folder)

    for file in sorted(path.iterdir()):
        if file.is_file():
            print(f"  {file.name} ({file.stat().st_size:,} bytes)")


data/
  bestbuy_page_reviews.csv (5,455 bytes)
  bestbuy_page_reviews.xlsx (7,865 bytes)
  hanuaI_reviews.xlsx (14,146 bytes)

reports/
  bestbuy_reviews_sentiment.csv (5,638 bytes)
  bestbuy_reviews_sentiment.xlsx (8,052 bytes)
  customer_theme_analysis.xlsx (5,323 bytes)
  HanuAI_Task1_Report.txt (3,497 bytes)
  preliminary_theme_counts.xlsx (4,989 bytes)
  review_analysis_summary.xlsx (5,002 bytes)
  review_insights.xlsx (18,117 bytes)
  scraping_inspection.xlsx (5,129 bytes)
  sentiment_results.xlsx (10,249 bytes)
  ~$review_insights.xlsx (165 bytes)


In [40]:
import pandas as pd

file_path = "data/hanuaI_reviews.xlsx"

xls = pd.ExcelFile(file_path)
print("Sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    data = pd.read_excel(file_path, sheet_name=sheet)

    print(f"\nSheet: {sheet}")
    print("Rows:", len(data))
    print("Columns:", list(data.columns))
    display(data.head(3))

    if "Source" in data.columns:
        print("\nSource counts:")
        print(data["Source"].value_counts(dropna=False))

Sheets: ['Sheet1']

Sheet: Sheet1
Rows: 50
Columns: ['ID', 'Review title', 'Reviewer', 'Date', 'Rating', 'Review Text', 'Source']


,ID,Review title,Reviewer,Date,Rating,Review Text,Source
0,REV001,Lux ANC Review - Great Color and Comfortable,Schulzy,2024-04-04,5,"Comfortable fit, attractive color, useful butt...",jlabaudio.com
1,REV002,THE MVP HEADPHONE!,TheRogueOne,2024-04-06,4,Positive overall experience.,jlabaudio.com
2,REV003,Great headphones on a budget,Jfrias1,2024-05-28,5,Good headphones after adjusting the EQ settings.,jlabaudio.com



Source counts:
Source
jlabaudio.com    49
bestbuy.com       1
Name: count, dtype: int64


In [41]:
import pandas as pd

df50 = pd.read_excel("data/hanuaI_reviews.xlsx")

required_columns = [
    "ID", "Review title", "Reviewer", "Date",
    "Rating", "Review Text", "Source"
]

print("Total rows:", len(df50))
print("Required columns present:", all(c in df50.columns for c in required_columns))
print("Missing values per column:\n", df50[required_columns].isna().sum())
print("Duplicate IDs:", df50["ID"].duplicated().sum())
print("Duplicate review texts:", df50["Review Text"].duplicated().sum())
print("Ratings outside 1–5:", (~df50["Rating"].between(1, 5)).sum())
print("Date parsing failures:", pd.to_datetime(df50["Date"], errors="coerce").isna().sum())
print("\nSource counts:\n", df50["Source"].value_counts(dropna=False))

Total rows: 50
Required columns present: True
Missing values per column:
 ID              0
Review title    0
Reviewer        0
Date            0
Rating          0
Review Text     0
Source          0
dtype: int64
Duplicate IDs: 0
Duplicate review texts: 0
Ratings outside 1–5: 0
Date parsing failures: 0

Source counts:
 Source
jlabaudio.com    49
bestbuy.com       1
Name: count, dtype: int64


In [42]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from pathlib import Path

nltk.download("vader_lexicon", quiet=True)

# Load the validated 50-row dataset
df = pd.read_excel("data/hanuaI_reviews.xlsx")

# Analyze the review text
sia = SentimentIntensityAnalyzer()

df["Sentiment Score"] = df["Review Text"].fillna("").astype(str).apply(
    lambda text: sia.polarity_scores(text)["compound"]
)

def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    return "Neutral"

df["Sentiment"] = df["Sentiment Score"].apply(classify_sentiment)

# Save detailed results
Path("reports").mkdir(exist_ok=True)

df.to_csv(
    "reports/50_review_sentiment_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)
df.to_excel(
    "reports/50_review_sentiment_analysis.xlsx",
    index=False
)

# Print the analysis summary
print("Reviews analyzed:", len(df))
print("\nSentiment distribution:")
print(df["Sentiment"].value_counts())

print("\nAverage star rating:", round(df["Rating"].mean(), 2), "/ 5")

print("\nSentiment by rating:")
print(pd.crosstab(df["Rating"], df["Sentiment"]))

print("\nSource counts:")
print(df["Source"].value_counts())

print("\nSaved 50-review sentiment results.")

Reviews analyzed: 50

Sentiment distribution:
Sentiment
Positive    36
Negative    10
Neutral      4
Name: count, dtype: int64

Average star rating: 3.54 / 5

Sentiment by rating:
Sentiment  Negative  Neutral  Positive
Rating                                
1                 4        2         1
2                 3        1         2
3                 2        1         5
4                 1        0        10
5                 0        0        18

Source counts:
Source
jlabaudio.com    49
bestbuy.com       1
Name: count, dtype: int64

Saved 50-review sentiment results.


In [43]:
import pandas as pd
from pathlib import Path

df = pd.read_excel("reports/50_review_sentiment_analysis.xlsx")

insights = pd.DataFrame([
    {
        "Finding": "Overall sentiment",
        "Evidence": f"{(df['Sentiment'] == 'Positive').sum()} of {len(df)} reviews are positive",
        "Recommendation": "Maintain product features that customers praise and monitor negative feedback."
    },
    {
        "Finding": "Negative feedback",
        "Evidence": f"{(df['Sentiment'] == 'Negative').sum()} reviews are negative",
        "Recommendation": "Manually review negative comments to identify specific product issues."
    },
    {
        "Finding": "Average rating",
        "Evidence": f"{df['Rating'].mean():.2f} out of 5",
        "Recommendation": "Track ratings over time and compare them with sentiment trends."
    },
    {
        "Finding": "Review source",
        "Evidence": "; ".join(
            f"{source}: {count}"
            for source, count in df["Source"].value_counts().items()
        ),
        "Recommendation": "Preserve source attribution and verify provenance before making Best Buy-specific claims."
    },
    {
        "Finding": "Sentiment method",
        "Evidence": "VADER rule-based classification",
        "Recommendation": "Manually validate a sample because text sentiment can differ from star ratings."
    }
])

Path("reports").mkdir(exist_ok=True)
insights.to_excel("reports/business_insights_50_reviews.xlsx", index=False)

display(insights)
print("Saved: reports/business_insights_50_reviews.xlsx")


,Finding,Evidence,Recommendation
0,Overall sentiment,36 of 50 reviews are positive,Maintain product features that customers prais...
1,Negative feedback,10 reviews are negative,Manually review negative comments to identify ...
2,Average rating,3.54 out of 5,Track ratings over time and compare them with ...
3,Review source,jlabaudio.com: 49; bestbuy.com: 1,Preserve source attribution and verify provena...
4,Sentiment method,VADER rule-based classification,Manually validate a sample because text sentim...


Saved: reports/business_insights_50_reviews.xlsx


In [44]:
import pandas as pd
from pathlib import Path

df = pd.read_excel("reports/50_review_sentiment_analysis.xlsx")

sentiment_counts = df["Sentiment"].value_counts()
source_counts = df["Source"].value_counts()

report = f"""
HANUAI – ML ASSESSMENT
Task 1: Customer Review Analysis

1. Objective
The objective is to collect and structure product reviews, analyze sentiment,
identify recurring themes, document collection challenges, and provide
business recommendations.

2. Product
JLab JBuds Lux Over-Ear Noise Cancelling True Wireless Headphones – Mauve

Product page:
https://www.bestbuy.ca/en-ca/product/jlab-jbuds-lux-over-ear-noise-cancelling-true-wireless-headphones-mauve-only-at-best-buy/18145791

3. Dataset and Provenance
The existing analysis dataset contains {len(df)} review records and the
required fields: ID, Review title, Reviewer, Date, Rating, Review Text, and Source.

Source counts:
{source_counts.to_string()}

The page's embedded state provided 10 records during the live-page inspection.
The 50-row analysis dataset was available as a separate Excel file. Its
collection history has not been independently verified, so the 50 records
must not be represented as 50 newly scraped Best Buy Canada reviews.

4. Data Quality
The 50-row dataset has no missing values, duplicate IDs, duplicate review
texts, invalid ratings, or unparseable dates according to the validation
checks performed.

5. Sentiment Analysis
VADER was used as a rule-based sentiment analyzer. Compound scores >= 0.05
were labelled Positive, scores <= -0.05 Negative, and other scores Neutral.

Results:
- Reviews analyzed: {len(df)}
- Positive: {int(sentiment_counts.get("Positive", 0))}
- Negative: {int(sentiment_counts.get("Negative", 0))}
- Neutral: {int(sentiment_counts.get("Neutral", 0))}
- Average star rating: {df["Rating"].mean():.2f} out of 5

VADER estimates sentiment from text and may disagree with numerical star ratings.
Manual review is recommended to evaluate classification quality.

6. Review Collection Challenges
- A direct HTTP request to the product page returned HTTP 403.
- The browser loaded the product page, but the inspected page state contained
  only 10 review records.
- The page reported additional reviews and pagination, but further records
  were not retrieved during this run.
- Review sorting and pagination controls were not clearly exposed in the
  inspected DOM.
- The required minimum of 50 reviews was analyzed using the separate existing
  dataset; this does not establish that 50 records were scraped from Best Buy.

No access controls were bypassed. Additional records should be obtained
through a permitted interface or an authorized data source.

7. Preliminary Business Insights
- {int(sentiment_counts.get("Positive", 0))} of {len(df)} reviews were classified as positive.
- {int(sentiment_counts.get("Negative", 0))} reviews were classified as negative.
- The average star rating was {df["Rating"].mean():.2f} out of 5.
- Negative comments should be reviewed to identify recurring customer concerns.
- Source attribution should be preserved when analyzing syndicated reviews.

8. Recommendations
- Manually inspect negative reviews and group them by recurring product issue.
- Track ratings and sentiment over time.
- Validate a sample of VADER predictions manually.
- Use an authorized and reproducible review collection method.
- Verify dataset provenance before making Best Buy-specific conclusions.

9. Deliverables
The working folder contains extracted review records, sentiment results,
preliminary theme analysis, business insights, and this report.

Outstanding limitation: the provenance of the 50-row analysis dataset has not
been independently verified as a 50-review extraction from Best Buy Canada.
"""

Path("reports/HanuAI_Task1_Report.txt").write_text(report, encoding="utf-8")
print("Updated report with 50-review analysis results.")


Updated report with 50-review analysis results.


In [45]:
import nbformat
from pathlib import Path

notebook_path = Path("HanuAI_Task1.ipynb")

if not notebook_path.exists():
    print("Notebook not found at:", notebook_path.resolve())
else:
    notebook = nbformat.read(notebook_path, as_version=4)

    code_cells = [
        cell for cell in notebook.cells
        if cell.cell_type == "code"
    ]

    print("Notebook:", notebook_path.resolve())
    print("Total cells:", len(notebook.cells))
    print("Code cells:", len(code_cells))
    print("Markdown cells:", sum(
        cell.cell_type == "markdown" for cell in notebook.cells
    ))

    errors = []
    for cell_number, cell in enumerate(code_cells, start=1):
        for output in cell.get("outputs", []):
            if output.output_type == "error":
                errors.append(
                    (cell_number, output.ename, output.evalue)
                )

    print("\nSaved execution errors:", len(errors))
    for error in errors:
        print(error)

    print("\nNotebook structure is valid:",
          nbformat.validate(notebook) is None)

Notebook: C:\Users\sridh\HanuAI_Task1\HanuAI_Task1.ipynb
Total cells: 44
Code cells: 44
Markdown cells: 0

Saved execution errors: 0

Notebook structure is valid: True


In [46]:
import nbformat
from pathlib import Path

notebook = nbformat.read("HanuAI_Task1.ipynb", as_version=4)

for i, cell in enumerate(notebook.cells, start=1):
    source = cell.source.strip().replace("\n", " ")
    preview = source[:180]
    print(f"\nCell {i} [{cell.cell_type}]")
    print(preview)


Cell 1 [code]
# HanuAI – ML Assessment Task 1 # Step 1: Import libraries and load the customer review dataset  import pandas as pd from pathlib import Path  # Define the project paths PROJECT_DI

Cell 2 [code]
# Step 2: Validate the customer review dataset  print("Total reviews:", len(df))  # Check for missing values print("\nMissing values in each column:") print(df.isnull().sum())  # C

Cell 3 [code]
# Step 3: Perform sentiment analysis using VADER  from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer  # Initialize the sentiment analyzer analyzer = SentimentInte

Cell 4 [code]
# Step 4: Save sentiment analysis results  from pathlib import Path  # Create the reports folder if it does not exist REPORTS_DIR = Path.cwd() / "reports" REPORTS_DIR.mkdir(exist_o

Cell 5 [code]
# Step 5: Analyze customer feedback themes  import re from collections import Counter  # Define the customer feedback themes and their keywords themes = {     "Sound Quality": ["so

Cell 6 [code]
# Ste

In [47]:
import nbformat

notebook = nbformat.read("HanuAI_Task1.ipynb", as_version=4)

print("Total cells:", len(notebook.cells))
print("Markdown cells:", sum(
    cell.cell_type == "markdown" for cell in notebook.cells
))
print("First cell type:", notebook.cells[0].cell_type)

Total cells: 50
Markdown cells: 1
First cell type: markdown


In [48]:
from pathlib import Path

required_files = [
    "HanuAI_Task1.ipynb",
    "data/hanuaI_reviews.xlsx",
    "data/bestbuy_page_reviews.csv",
    "reports/50_review_sentiment_analysis.csv",
    "reports/50_review_sentiment_analysis.xlsx",
    "reports/business_insights_50_reviews.xlsx",
    "reports/customer_theme_analysis.xlsx",
    "reports/scraping_inspection.xlsx",
    "reports/HanuAI_Task1_Report.txt",
]

print("HANUAI ASSESSMENT — DELIVERABLE CHECK\n")

missing = []

for filename in required_files:
    path = Path(filename)
    if path.is_file():
        print(f"[OK]      {filename}")
    else:
        print(f"[MISSING] {filename}")
        missing.append(filename)

print("\nMissing files:", len(missing))

HANUAI ASSESSMENT — DELIVERABLE CHECK

[OK]      HanuAI_Task1.ipynb
[OK]      data/hanuaI_reviews.xlsx
[OK]      data/bestbuy_page_reviews.csv
[OK]      reports/50_review_sentiment_analysis.csv
[OK]      reports/50_review_sentiment_analysis.xlsx
[OK]      reports/business_insights_50_reviews.xlsx
[OK]      reports/customer_theme_analysis.xlsx
[OK]      reports/scraping_inspection.xlsx
[OK]      reports/HanuAI_Task1_Report.txt

Missing files: 0
